In [1]:
import os
#import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.optimizers import RMSprop  
from tensorflow.keras.models import model_from_json
from tensorflow.keras.models import load_model

In [2]:
#Directory with angry images
Angry_dir= os.path.join(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy\angry")

# Directory with Happy images
Happy_dir= os.path.join(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy\happy")

# Directory with Neutral images
Neutral_dir = os.path.join(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy\neutral")

# Directory with Sad images
Sad_dir = os.path.join(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy\sad")

# Directory with Surprise images
Surprise_dir = os.path.join(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy\surprise")


In [3]:
train_Angry_names = os.listdir(Angry_dir)
print(train_Angry_names[:5])

train_Happy_names = os.listdir(Happy_dir)
print(train_Happy_names[:5])

#entire data is divided into batches
batch_size = 16

# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(rescale=1./255)

['0.jpg', '1.jpg', '10.jpg', '10002.jpg', '10016.jpg']
['100.jpg', '10000.jpg', '10001.jpg', '10005.jpg', '10035.jpg']


In [4]:
#properties of data

# Flow training images in batches of 128 using train_datagen generator
train_generator = train_datagen.flow_from_directory(r"C:\Users\Mohana\Desktop\DE ML\Datasets\happy",  # This is the source directory for training images
        target_size=(48, 48),  # All images will be resized to 48 x 48
        batch_size=batch_size,
        color_mode='grayscale',
        # Specify the classes explicitly
        classes = ['Angry','Happy','Neutral','Sad','Surprise'],
        # Since we use categorical_crossentropy loss, we need categorical labels
        class_mode='categorical')
target_size=(48,48)

Found 24282 images belonging to 5 classes.


In [5]:
#setting of algorithm

model = tf.keras.models.Sequential([
    # Note the input shape is the desired size of the image 48*48 with 3 bytes color

     # The first convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(48, 48, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    # The second convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    # The second convolution####
    ###tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    ###tf.keras.layers.MaxPooling2D(2,2),
    
    # The third convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    # Flatten the results to feed into a dense layer
    tf.keras.layers.Flatten(),
    # 64 neuron in the fully-connected layer
    tf.keras.layers.Dense(64, activation='relu'),
    
        
    # 5 output neurons for 5 classes with the softmax activation
    tf.keras.layers.Dense(5,activation='softmax')
])
model.summary()

C:\Users\Mohana\tensorflow_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 46, 46, 64)          │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 23, 23, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 21, 21, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 10, 10, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 8, 64)            │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 4, 4, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │          65,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 140,421 (548.52 KB)

 Trainable params: 140,421 (548.52 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:

model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['acc'])#RMSprop(lr=0.001)
# Total sample count

total_sample=train_generator.n



# Training
num_epochs = 5
model.fit(train_generator,steps_per_epoch=int(total_sample/batch_size),
                    epochs=num_epochs,verbose=1)


# Saving the model
# serialize model to JSON
model_json = model.to_json()
with open("modelGG.json", "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("model1GG.weights.h5")
print("Saved model to disk")

Epoch 1/5
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 72s 46ms/step - acc: 0.6221 - loss: 0.9498
Epoch 2/5
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 0s 11us/step - acc: 0.6250 - loss: 0.8836 
Epoch 3/5
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 63s 41ms/step - acc: 0.6445 - loss: 0.8994
Epoch 4/5
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 0s 12us/step - acc: 0.6250 - loss: 0.9482 
Epoch 5/5
1517/1517 ━━━━━━━━━━━━━━━━━━━━ 64s 42ms/step - acc: 0.6647 - loss: 0.8539
Saved model to disk
